In [1]:
import os
import requests
import time
import math
from dataclasses import dataclass
import pickle
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertForSequenceClassification

# --- 0. Setup ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # Use GPU if available, else CPU
print(f"Using device: {DEVICE}")

# --- 1. GPT Model Definition & Loading ---

# GPT Hyperparameters (must match training)
GPT_BLOCK_SIZE = 256
GPT_N_EMBD = 384
GPT_N_HEAD = 6
GPT_N_LAYER = 6
GPT_DROPOUT = 0.2

# --- 1.1 GPT Character-level Tokenizer ---
gpt_data_file_path = 'fake_news_formatted.txt' # Path to GPT's training data for vocab
if not os.path.exists(gpt_data_file_path):
    print(f"Error: GPT data file {gpt_data_file_path} not found.")
    exit()
with open(gpt_data_file_path, 'r', encoding='utf-8') as f:
    gpt_data_text = f.read()
gpt_chars = sorted(list(set(gpt_data_text)))
gpt_vocab_size = len(gpt_chars)
gpt_stoi = {ch: i for i, ch in enumerate(gpt_chars)} # char to int
gpt_itos = {i: ch for i, ch in enumerate(gpt_chars)} # int to char
def gpt_encode(s): return [gpt_stoi[c] for c in s]
def gpt_decode(l): return ''.join([gpt_itos[i] for i in l])
print(f"GPT Vocabulary size: {gpt_vocab_size}")

# --- 1.2 GPT Model Architecture ---
@dataclass
class GPTConfig: # Stores GPT model parameters
    block_size: int = GPT_BLOCK_SIZE
    vocab_size: int = gpt_vocab_size
    n_layer: int = GPT_N_LAYER
    n_head: int = GPT_N_HEAD
    n_embd: int = GPT_N_EMBD
    dropout: float = GPT_DROPOUT
    bias: bool = True

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module): # Transformer's attention mechanism
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias) # Q, K, V projections
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias) # Output projection
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size)) # Causal mask
    def forward(self, x):
        B, T, C = x.size()
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module): # Feed-forward network in Transformer block
    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module): # Single Transformer block (Attention + MLP)
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x)) # Residual connection around attention
        x = x + self.mlp(self.ln_2(x))   # Residual connection around MLP
        return x

class GPT(nn.Module): # The full GPT language model
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),      # Token embeddings
            wpe = nn.Embedding(config.block_size, config.n_embd),     # Position embeddings
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]), # Stack of N_LAYER blocks
            ln_f = LayerNorm(config.n_embd, bias=config.bias),        # Final LayerNorm
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) # Output layer
        self.transformer.wte.weight = self.lm_head.weight # Weight tying
        self.apply(self._init_weights) # Initialize weights
        for pn, p in self.named_parameters(): # Specific init for residual projections
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    def _init_weights(self, module): # Weight initialization
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None): # Model forward pass
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Input sequence length {t} exceeds block size {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None: # Calculate loss if targets are provided (training)
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else: # For inference, only compute logits for the last token
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None): # Text generation
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx

# --- 1.3 Load trained GPT model weights ---
gpt_model_config = GPTConfig()
gpt_model = GPT(gpt_model_config)
gpt_model_path = 'GPTmodel_trained_Newdataset.pth' # Path to your trained GPT model

if os.path.exists(gpt_model_path):
    gpt_model.load_state_dict(torch.load(gpt_model_path, map_location=DEVICE, weights_only=True))
    gpt_model.to(DEVICE)
    gpt_model.eval()
    print(f"GPT Model loaded successfully from {gpt_model_path}")
else:
    print(f"GPT Model file not found: {gpt_model_path}. Cannot generate text.")
    exit()

# --- 2. Generate text using GPT ---
print(f"\n--- Generating text using GPT ---")
start_string = "[input] " # Prompt for GPT
print(f"Starting prompt: '{start_string.strip()}'")
start_ids = gpt_encode(start_string)
x_input_gpt = torch.tensor(start_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

with torch.no_grad(): # Generate text
    generated_ids = gpt_model.generate(x_input_gpt, max_new_tokens=200, temperature=0.7, top_k=50)
generated_text_gpt = gpt_decode(generated_ids[0].tolist())

# Clean up generated text (remove prompt, potentially split if GPT generates helper tokens)
clean_generated_text = generated_text_gpt.replace(start_string, "").strip()
parts = clean_generated_text.split("[label]") # Example: if GPT learned to output "[label]"
final_generated_text_for_bert = parts[0].strip() if len(parts) > 0 else clean_generated_text

print("\n--- GPT Generated Text (for BERT classification) ---")
print(final_generated_text_for_bert)
print("="*50)



Using device: cpu
GPT Vocabulary size: 275
GPT Model loaded successfully from GPTmodel_trained_Newdataset.pth

--- Generating text using GPT ---
Starting prompt: '[input]'

--- GPT Generated Text (for BERT classification) ---
Texas indicate state of state to protect Trump on Friday evening to agree && WASHINGTON (Reuters) - The U.S. Senate on Tuesday a react for the comment on Tuesday after Russian citizen is killing in Ju


In [ ]:
# --- 3. Load BERT Classifier Model and Tokenizer ---
bert_model_save_path = 'best_bert_model.pt' # Path to your trained BERT classifier
bert_tokenizer_path = '../bert_model'       # Path to BERT tokenizer files (or 'bert-base-uncased')

if not os.path.exists(bert_model_save_path):
    print(f"BERT model file {bert_model_save_path} not found. Cannot classify.")
    exit()

bert_tokenizer = BertTokenizer.from_pretrained(bert_tokenizer_path, local_files_only=True)
print(f"BERT Tokenizer loaded from local path: {bert_tokenizer_path}")

bert_classifier_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2) # 2 labels for binary classification

bert_classifier_model.load_state_dict(torch.load(bert_model_save_path, map_location=DEVICE, weights_only=True))

bert_classifier_model.to(DEVICE)
bert_classifier_model.eval()
print(f"BERT Classifier Model loaded successfully from {bert_model_save_path}")

# --- 4. Preprocess GPT-generated text for BERT ---
bert_encodings = bert_tokenizer( # Tokenize text for BERT
    [final_generated_text_for_bert], # Must be a list
    padding=True, truncation=True, max_length=128, return_tensors='pt' # BERT specific settings
)
input_ids_bert = bert_encodings['input_ids'].to(DEVICE)
attention_mask_bert = bert_encodings['attention_mask'].to(DEVICE)

# --- 5. Classify with BERT model ---
print("\n--- Classifying with BERT ---")
with torch.no_grad(): # Inference mode
    outputs_bert = bert_classifier_model(input_ids=input_ids_bert, attention_mask=attention_mask_bert)
    logits_bert = outputs_bert.logits
    probabilities_bert = F.softmax(logits_bert, dim=1)
    predicted_class_bert = torch.argmax(probabilities_bert, dim=1)

# --- 6. Display classification result ---
class_labels = {0: "Real News (Class 0)", 1: "Fake News (Class 1)"} # Adjust labels as needed
predicted_label_name = class_labels.get(predicted_class_bert.item(), "Unknown Class")

print(f"\nGenerated Text (Snippet): \"{final_generated_text_for_bert[:200]}...\"")
print(f"Predicted Class ID by BERT: {predicted_class_bert.item()}")
print(f"Predicted Label by BERT: {predicted_label_name}")
print(f"Probabilities: Class 0 = {probabilities_bert[0][0].item():.4f}, Class 1 = {probabilities_bert[0][1].item():.4f}")